#**AML LAB**

##**Experiment 9**

### Implement a classification model to distinguish between normal and abnormal physiological signals using extracted signal features. (Perform comparative analysis on different ML models)
### Dataset Link: https://www.kaggle.com/datasets/shayanfazeli/heartbeat


###1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

###2. Load Datasets

In [ ]:
# MIT-BIH
train = pd.read_csv("mitbih_train.csv", header=None)
test = pd.read_csv("mitbih_test.csv", header=None)

# PTBDB
normal = pd.read_csv("ptbdb_normal.csv", header=None)
abnormal = pd.read_csv("ptbdb_abnormal.csv", header=None)

###3. Preprocessing

In [ ]:
X_train_mit = train.iloc[:, :-1]
y_train_mit = train.iloc[:, -1]

X_test_mit = test.iloc[:, :-1]
y_test_mit = test.iloc[:, -1]

# Convert multi-class → binary
y_train_mit = y_train_mit.apply(lambda x: 0 if x == 0 else 1)
y_test_mit = y_test_mit.apply(lambda x: 0 if x == 0 else 1)


df_ptb = pd.concat([normal, abnormal])

X_ptb = df_ptb.iloc[:, :-1]
y_ptb = df_ptb.iloc[:, -1]

# Train-test split
X_train_ptb, X_test_ptb, y_train_ptb, y_test_ptb = train_test_split(
    X_ptb, y_ptb, test_size=0.2, random_state=42
)

###4. Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train_mit = scaler.fit_transform(X_train_mit)
X_test_mit = scaler.transform(X_test_mit)

X_train_ptb = scaler.fit_transform(X_train_ptb)
X_test_ptb = scaler.transform(X_test_ptb)

###5. Define Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "SVM": SVC()
}

###6. Training + Evaluation Function

In [ ]:
def evaluate_models(X_train, X_test, y_train, y_test, dataset_name):
    results = []

    print(f"\n===== {dataset_name} Results =====\n")

    for name, model in models.items():
        try:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            acc = accuracy_score(y_test, y_pred)

            print(f"{name}:")
            print("Accuracy:", acc)
            print(classification_report(y_test, y_pred))
            print(confusion_matrix(y_test, y_pred))
            print("-" * 50)

            results.append((name, acc))
        except Exception as e:
            print(f"{name}: Failed to evaluate due to error: {e}")
            print("-" * 50)
            results.append((name, np.nan)) # Append NaN for accuracy if evaluation fails

    return pd.DataFrame(results, columns=["Model", "Accuracy"])

###7. Run Experiment

In [ ]:
mit_results = evaluate_models(
    X_train_mit, X_test_mit,
    y_train_mit, y_test_mit,
    "MIT-BIH"
)

ptb_results = evaluate_models(
    X_train_ptb, X_test_ptb,
    y_train_ptb, y_test_ptb,
    "PTBDB"
)


===== MIT-BIH Results =====

Logistic Regression:
Accuracy: 0.9046226932212681
              precision    recall  f1-score   support

           0       0.91      0.98      0.94     18118
           1       0.84      0.55      0.67      3774

    accuracy                           0.90     21892
   macro avg       0.88      0.76      0.80     21892
weighted avg       0.90      0.90      0.90     21892

[[17727   391]
 [ 1697  2077]]
--------------------------------------------------
KNN:
Accuracy: 0.9766124611730312
              precision    recall  f1-score   support

           0       0.98      0.99      0.99     18118
           1       0.97      0.90      0.93      3774

    accuracy                           0.98     21892
   macro avg       0.97      0.94      0.96     21892
weighted avg       0.98      0.98      0.98     21892

[[17999   119]
 [  393  3381]]
--------------------------------------------------
Decision Tree:
Accuracy: 0.9596199524940617
              precision 

###8. Comparative Analysis

In [ ]:
print("MIT-BIH Comparison:")
print(mit_results)

print("\nPTBDB Comparison:")
print(ptb_results)

MIT-BIH Comparison:
                 Model  Accuracy
0  Logistic Regression  0.904623
1                  KNN  0.976612
2        Decision Tree  0.959620
3        Random Forest  0.977526
4                  SVM  0.970126

PTBDB Comparison:
                 Model  Accuracy
0  Logistic Regression  0.833734
1                  KNN  0.929577
2        Decision Tree  0.925455
3        Random Forest  0.972175
4                  SVM  0.918928
